In [29]:
from mainnet_launch.pages.asset_discounts.stable_coin_pricing import (
    StableCoinConsants,
    stablecoin_constants,
    stables,
    usde_constants,
    sUSDe,
    USDC,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    get_state_by_one_block,
    build_blocks_to_use,
)
from mainnet_launch.constants import ETH_CHAIN
import plotly.express as px
import plotly.io as pio

# by def USDC is always at peg.the wrost safe price is 2 bips
pio.templates.default = None
token_symbols = [c.symbol for c in stablecoin_constants]
backing_calls = [c.backing_call for c in stablecoin_constants]
safe_price_calls = [c.safe_price_call for c in stablecoin_constants if c.safe_price_call is not None]
blocks = build_blocks_to_use(ETH_CHAIN, start_block=21389810)
df = get_raw_state_by_blocks([*backing_calls, *safe_price_calls], blocks, ETH_CHAIN, include_block_number=True)
df["sUSDs_safe_price"] = df["sUSDs_backing"] * df["USDs_safe_price"]
df["scrvUSD_safe_price"] = df["scrvUSD_backing"] * df["crvUSD_safe_price"]
df["sUSDe_safe_price"] = df["sUSDe_backing"] * df["USDe_safe_price"]

# 100 * (peg - safe) / peg

safe_price_cols = [c for c in df.columns if "_safe_price" in c]
px.line(df[safe_price_cols])

Inserted new rows into 'BLOCKS_TO_USE_TABLE' while avoiding duplicates.
Table 'BLOCKS_TO_USE_TABLE'already exists and data inserted.
Successfully updated table_name='BLOCKS_TO_USE_TABLE' current_time=datetime.datetime(2025, 3, 12, 0, 14, 43, 754277, tzinfo=datetime.timezone.utc).
Inserted new rows into 'BLOCKS_TO_USE_TABLE' while avoiding duplicates.
Table 'BLOCKS_TO_USE_TABLE'already exists and data inserted.
Successfully updated table_name='BLOCKS_TO_USE_TABLE' current_time=datetime.datetime(2025, 3, 12, 0, 14, 44, 821266, tzinfo=datetime.timezone.utc).


In [2]:
spot_price_calls = [call for calls in [c.spot_price_calls for c in stablecoin_constants] for call in calls]
spot_price_calls
spot_price_df = get_raw_state_by_blocks(spot_price_calls[:1], [min(blocks), max(blocks)], ETH_CHAIN)
# spot_price_df

In [3]:
for symbol in token_symbols:
    df[f"{symbol}_discount"] = 100 * ((df[f"{symbol}_backing"] - df[f"{symbol}_safe_price"]) / df[f"{symbol}_backing"])

discount_cols = [c for c in df.columns if "_discount" in c]
px.line(df[discount_cols], title="stablecoin percent discoutns")

In [5]:
spot_price_calls = [call for calls in [c.spot_price_calls for c in stablecoin_constants] for call in calls]
spot_price_df = get_raw_state_by_blocks(spot_price_calls, blocks, ETH_CHAIN)

px.line(spot_price_df)

In [7]:
spot_price_df.head(1).T

timestamp,2024-12-12 23:48:23+00:00
DAI_spot_to_USDC,1.000399
DAI_spot_to_USDT,0.999611
USDe_spot_to_USDC,0.000000
USDe_spot_to_USDT,0.000000
USDC_spot_to_USDC,0.000000
USDC_spot_to_USDT,1.012509
USDT_spot_to_USDC,1.000738
USDT_spot_to_USDT,0.000000
USDs_spot_to_USDC,0.000000
USDs_spot_to_USDT,0.000000


In [ ]:
# GHO_to_crvUSD_spot_price = Call(
#     "0x635EF0056A597D13863B73825CcA297236578595",
#     ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
#     [("GHO_TO_crvUSD_spot_price", safe_normalize_with_bool_success)],
# )

# GHO_to_USDe_spot_price = Call(
#     "0x670a72e6D22b0956C0D2573288F82DCc5d6E3a61",
#     ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
#     [("GHO_to_USDe_spot_price", safe_normalize_with_bool_success)],
# )

In [25]:
from multicall import Call

from mainnet_launch.data_fetching.get_state_by_block import (
    safe_normalize_6_with_bool_success,
    safe_normalize_with_bool_success,
)


DAI_to_USDC_spot_price = Call(
    "0xbEbc44782C7dB0a1A60Cb6fe97d0b483032FF1C7",
    ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
    [("DAI_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)

DAI_to_USDC_spot_price2 = Call(
    "0xA5407eAE9Ba41422680e2e00537571bcC53efBfD",
    ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
    [("DAI_to_USDC_spot_price2", safe_normalize_6_with_bool_success)],
)


USDT_to_USDC_spot_price = Call(
    "0xbEbc44782C7dB0a1A60Cb6fe97d0b483032FF1C7",
    ["get_dy(int128,int128,uint256)(uint256)", 2, 1, int(1e6)],
    [("USDT_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)

USDT_to_USDC_spot_price_2 = Call(
    "0xA5407eAE9Ba41422680e2e00537571bcC53efBfD",
    ["get_dy(int128,int128,uint256)(uint256)", 2, 1, int(168)],
    [("USDT_to_USDC_spot_price_2", safe_normalize_6_with_bool_success)],
)


crvUSD_to_USDC_spot_price = Call(
    "0x4DEcE678ceceb27446b35C672dC7d61F30bAD69E",
    ["get_dy(int128,int128,uint256)(uint256)", 1, 0, int(1e18)],
    [("crvUSD_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)


crvUSD_to_USDT_spot_price = Call(
    "0x390f3595bCa2Df7d23783dFd126427CCeb997BF4",
    ["get_dy(int128,int128,uint256)(uint256)", 1, 0, int(1e18)],
    [("crvUSD_to_USDT_spot_price", safe_normalize_6_with_bool_success)],
)


USDe_to_USDC_spot_price = Call(
    "0x02950460E2b9529D0E00284A5fA2d7bDF3fA4d72",
    ["get_dy(int128,int128,uint256)(uint256)", 0, 1, int(1e18)],
    [("USDe_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)


# sUSDe -> USDC balancer
# https://balancer.fi/pools/ethereum/v2/0xb819feef8f0fcdc268afe14162983a69f6bf179e000000000000000000000689


sUSD_to_USDC_spot_price = Call(
    "0xA5407eAE9Ba41422680e2e00537571bcC53efBfD",
    ["get_dy(int128,int128,uint256)(uint256)", 3, 1, int(1e18)],
    [("sUSD_to_USDC_spot_price", safe_normalize_6_with_bool_success)],
)


# balancer GHO -> USDC
# https://balancer.fi/pools/ethereum/v2/0x8353157092ed8be69a9df8f95af097bbf33cb2af0000000000000000000005d9


# https://balancer.fi/pools/ethereum/v2/0x8353157092ed8be69a9df8f95af097bbf33cb2af0000000000000000000005d9
# USDT, GHO, USDC


calls = [
    crvUSD_to_USDC_spot_price,
    USDe_to_USDC_spot_price,
    DAI_to_USDC_spot_price,
    USDT_to_USDC_spot_price,
]

get_state_by_one_block(calls, max(blocks), ETH_CHAIN)

{'crvUSD_to_USDC_spot_price': 0.999437,
 'USDe_to_USDC_spot_price': 0.998997,
 'DAI_to_USDC_spot_price': 0.999727,
 'USDT_to_USDC_spot_price': 0.999346,
 'GHO_TO_crvUSD_spot_price': 0.999817102566477,
 'GHO_to_USDe_spot_price': 1.000390437097556}

In [8]:
# s  # sUSDe -> usdc https://balancer.fi/pools/ethereum/v2/0xb819feef8f0fcdc268afe14162983a69f6bf179e000000000000000000000689

# gho, USDC, USDT tri pool  https://balancer.fi/pools/ethereum/v2/0x8353157092ed8be69a9df8f95af097bbf33cb2af0000000000000000000005d9

# https://curve.fi/dex/ethereum/pools/factory-crvusd-0/deposit/
# crvUSD -> USDC


# dai, USDC, USDT tri pool
# https://curve.fi/dex/ethereum/pools/3pool/deposit/

# USDe -> USDC https://curve.fi/dex/ethereum/pools/factory-stable-ng-12/deposit/

# USDs does not have good sorces of liquidty


# https://app.uniswap.org/explore/pools/ethereum/0x5777d92f208679DB4b9778590Fa3CAB3aC9e2168

NameError: name 's' is not defined

In [44]:
# from web3 import Web3

# # Assume you have an initialized Web3 instance (e.g. using an Infura or Alchemy endpoint)

# queries_address = Web3.toChecksumAddress("0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5")

# pool_id = bytes.fromhex("0xb819feef8f0fcdc268afe14162983a69f6bf179e000000000000000000000689"[2:])
# amount_in = int(1e18)
# swap_kind = 0
# user_data = b""

# from_internal_balance = False
# to_internal_balance = False

# call_obj = Call(
#     "0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5",  # Balancer Queries contract address
#     [
#         # Function signature for querySwap with two tuple parameters:
#         "querySwap((bytes32,uint8,address,address,uint256,bytes),(address,bool,address,bool))(uint256)",
#         pool_id,
#         swap_kind,
#         sUSDe,
#         USDC,
#         amount_in,
#         user_data,
#         '0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5',
#         from_internal_balance,
#         '0xE39B5e3B6D74016b2F6A9673D7d7493B6DF549d5',
#         to_internal_balance
#     ]
#     [("usdc_out", safe_normalize_6_with_bool_success)]
# )

# get_state_by_one_block([call_obj],22027082, ETH_CHAIN )

<>:17: SyntaxWarning:

list indices must be integers or slices, not tuple; perhaps you missed a comma?

<>:17: SyntaxWarning:

list indices must be integers or slices, not tuple; perhaps you missed a comma?

/tmp/ipykernel_75681/2292864164.py:17: SyntaxWarning:

list indices must be integers or slices, not tuple; perhaps you missed a comma?

/tmp/ipykernel_75681/2292864164.py:17: SyntaxWarning:

list indices must be integers or slices, not tuple; perhaps you missed a comma?

/tmp/ipykernel_75681/2292864164.py:17: SyntaxWarning:

list indices must be integers or slices, not tuple; perhaps you missed a comma?



TypeError: list indices must be integers or slices, not tuple